# НИРС по анализу данных.

## Построение модели машинного обучения для решения задач классификации и регрессии.

В качестве набора данных мы будем использовать набор данных 
по ожирению у людей - https://www.kaggle.com/datasets/adeniranstephen/obesity-prediction-dataset?resource=download

Эта задача является очень актуальной в условиях современности: мировые правительства (министерства здоровья РФ, ЕС, США в особенности) очень озабочены проблемой ожирения населения даже с самого малого возраста. В связи с этим я берусь за задачу построения модели, которая выявит тенденции и найдет наибольшие предикторы ожирения.

Датасет состоит из трех файлов:
- obesity_train.csv - обучающая выборка
- obesity_test.csv - тестовая выборка
- obesity_test_full.csv - полный датасет для теста

Каждый файл содержит следующие колонки:
- Gender (Пол) - бинарный признак: Male (мужской), Female (женский).
- Age (Возраст) - возраст человека в годах.
- Height (Рост) - рост в метрах.
- Weight (Вес) - вес в килограммах.
- family_history_with_overweight (Наличие семейной истории избыточного веса) - yes/no (да/нет).
- FAVC (Частое употребление высококалорийной пищи) - yes/no (да/нет).
- FCVC (Частота употребления овощей) - числовое значение (шкала от 1 до 3).
- NCP (Количество основных приемов пищи в день) - числовое значение (например, 1-4).
- CAEC (Употребление пищи между приемами пищи) - категориальный признак: Sometimes (иногда), Frequently (часто), no (нет) и др.
- SMOKE (Курение) - yes/no (да/нет).
- CH2O (Потребление воды в день) - числовое значение (шкала 1-3).
- SCC (Мониторинг потребления калорий) - yes/no (да/нет).
- FAF (Физическая активность в неделю) - числовое значение (шкала 0-3).
- TUE (Время использования электронных устройств) - числовое значение (шкала 0-3).
- CALC (Употребление алкоголя) - категориальный признак: Sometimes (иногда), Frequently (часто), no (нет).
- MTRANS (Способ передвижения) - категориальный признак: Public_Transportation (общественный транспорт), Walking (ходьба), Automobile (автомобиль), Motorbike (мотоцикл).
- Obesity (Уровень ожирения) - целевая переменная, категориальный признак: Normal_Weight (нормальный вес), Overweight_Level_I (избыточный вес I степени), Overweight_Level_II (избыточный вес II степени) и др.

В рассматриваемом примере будем решать обе задачи - и задачу классификации, и задачу регрессии:
- Для решения **задачи классификации** в качестве целевого признака будем использовать "Obesity".
- Для решения **задачи регрессии** в качестве целевого признака будем использовать "BMI" - индекс массы тела Weight / (Height^2), который вычислим для датасета позже.

### Импорт библиотек
Импортируем библиотеки с помощью команды import. Как правило, все команды import размещают в первых ячейках ноутбука.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, median_absolute_error, r2_score 
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.svm import SVC, NuSVC, LinearSVC, OneClassSVM, SVR, NuSVR, LinearSVR
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_graphviz
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
%matplotlib inline 
sns.set(style="ticks")

### Загрузка данных

Загрузим файлы датасета в помощью библиотеки Pandas. 

In [ ]:
# Обучающая выборка
original_train = pd.read_csv('data/obesity_train.csv', sep=",")
# Тестовая выборка (два файла)
original_test_1 = pd.read_csv('data/obesity_test.csv', sep=",")
original_test_2 = pd.read_csv('data/obesity_test_full.csv', sep=",")

In [ ]:
# Удалим дубликаты записей, если они присутствуют
train = original_train.drop_duplicates()
test_1 = original_test_1.drop_duplicates()
test_2 = original_test_2.drop_duplicates()

## Проведение разведочного анализа данных. Построение графиков, необходимых для понимания структуры данных. Анализ и заполнение пропусков в данных.

### Основные характеристики датасетов

In [ ]:
# Первые 5 строк датасета
train.head()

In [ ]:
# Размер обучающего датасета - 8143 строк, 7 колонок
train.shape, test_1.shape, test_2.shape

In [ ]:
# Список колонок
train.columns

In [ ]:
# Список колонок с типами данных 
# убедимся что типы данных одинаковы в обучающей и тестовых выборках
train.dtypes

In [ ]:
# Проверим наличие пустых значений
train.isnull().sum()

In [ ]:
test_1.isnull().sum()

In [ ]:
test_2.isnull().sum()

**Вывод. Представленный набор данных не содержит пропусков ни в обучающей, ни в тестовой выборках.**

### Feature Engineering

In [ ]:
# Feature engineering function
def add_features(df):
    df = df.assign(
        # 1. Metabolic risk (now ranges 0-4 instead of binary)
        Metabolic_Risk_Score = (df['FAVC'] * 2) + (df['FAF'] < 2).astype(int) + (df['TUE'] > 2).astype(int) + 0.5,
        
        #2.
        Activity_Score = (5 - df['FAF']) + (3 - df['TUE']) + (5 - df['MTRANS']) - 4,
        
        # 3. New suggested features
        Diet_Quality = df['FCVC'] - df['FAVC'],  # Veggies vs junk food
        Sedentary_Index = (df['FAF'] + df['TUE']) / 2
    )
    return df

In [ ]:
# Apply to all datasets
train = add_features(train)
test_1 = add_features(test_1)
test_2 = add_features(test_2)

### Построение графиков для понимания структуры данных

In [ ]:
# Парные диаграммы
sns.pairplot(train)

In [ ]:
sns.pairplot(train, hue="Obesity")

In [ ]:
# Оценим дисбаланс классов для Occupancy
fig, ax = plt.subplots(figsize=(2,2)) 
plt.hist(train['Obesity'])
plt.show()

train['Obesity'].value_counts()

In [ ]:
# посчитаем дисбаланс классов
total = train.shape[0]
classes = []
class_n=0
for item in train['Obesity'].value_counts():
    classes.append(item)
    print('Класс {} составляет {}%.'
      .format(class_n+1, round(classes[class_n] / total, 4)*100))
    class_n+=1
classes

**Вывод. Дисбаланс классов присутствует, но является приемлемым.**

In [ ]:
train.columns

In [ ]:
# Скрипичные диаграммы для числовых колонок
for col in ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight',
       'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE',
       'CALC', 'MTRANS', 'BMI', 'Obesity', 'Metabolic_Risk_Score', 'Activity_Score']:
    sns.violinplot(x=train[col])
    plt.show()

## Выбор признаков, подходящих для построения моделей. Кодирование категориальных признаков. Масштабирование данных. Формирование вспомогательных признаков, улучшающих качество моделей.

Для построения моделей будем использовать все признаки.

Категориальные признаки присутствуют, требуется кодирование некоторых из них.

Построим некоторые вспомогательные признаки.

Выполним масштабирование данных. Для этого необходимо объединить обучающую и тестовые выборки.

In [ ]:
# Создадим вспомогательные колонки, 
# чтобы наборы данных можно было разделить.
train['dataset'] = 'TRAIN'
test_1['dataset'] = 'TEST1'
test_2['dataset'] = 'TEST2'

In [ ]:
# Колонки для объединения
join_cols = ['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight',
       'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE',
       'CALC', 'MTRANS', 'BMI', 'Obesity', 'Metabolic_Risk_Score', 'Activity_Score', 'dataset']

In [ ]:
data_all = pd.concat([train[join_cols], test_1[join_cols], test_2[join_cols]])

In [ ]:
# Проверим корректность объединения
assert data_all.shape[0] == train.shape[0]+test_1.shape[0]+test_2.shape[0]

In [ ]:
data_all.head()

In [ ]:
# Числовые колонки для масштабирования
scale_cols = ['Height', 'Weight', 
       'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE',
       'CALC', 'MTRANS', 'CAEC']

In [ ]:
sc1 = MinMaxScaler()
sc1_data = sc1.fit_transform(data_all[scale_cols])

In [ ]:
# Добавим масштабированные данные в набор данных
for i in range(len(scale_cols)):
    col = scale_cols[i]
    new_col_name = col + '_scaled'
    data_all[new_col_name] = sc1_data[:,i]

In [ ]:
data_all.head()

In [ ]:
# Проверим, что масштабирование не повлияло на распределение данных
for col in scale_cols:
    col_scaled = col + '_scaled'

    fig, ax = plt.subplots(1, 2, figsize=(8,3))
    ax[0].hist(data_all[col], 50)
    ax[1].hist(data_all[col_scaled], 50)
    ax[0].title.set_text(col)
    ax[1].title.set_text(col_scaled)
    plt.show()

### Финальный Feature Engineering

In [ ]:
data_all['Age_Activity'] = data_all['Age'] * data_all['FAF_scaled']
data_all['Diet_Quality'] = data_all['FCVC_scaled'] - data_all['FAVC']  # Veggies vs junk food
data_all['Sedentary_Index'] = (data_all['FAF_scaled'] + data_all['TUE_scaled']) / 2

## Проведение корреляционного анализа данных. Формирование промежуточных выводов о возможности построения моделей машинного обучения. 

In [ ]:
# Воспользуемся наличием тестовых выборок, 
# включив их в корреляционную матрицу
corr_cols_1 = scale_cols + ['Obesity'] + ['BMI']
corr_cols_1

In [ ]:
scale_cols_postfix = [x+'_scaled' for x in scale_cols]
corr_cols_2 = scale_cols_postfix + ['Obesity'] + ['BMI']
corr_cols_2

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(data_all[corr_cols_1].corr(), annot=True, fmt='.2f')
ax.set_title('Исходные данные (до масштабирования)')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
sns.heatmap(data_all[corr_cols_2].corr(), annot=True, fmt='.2f')
ax.set_title('Масштабированные данные')
plt.show()

На основе корреляционной матрицы можно сделать следующие выводы:

- Корреляционные матрицы для исходных и масштабированных данных совпадают.
- Целевой признак классификации "Obesity" наиболее сильно коррелирует с весом (0.70), историей семьи (0.50). Эти признаки обязательно следует оставить в модели классификации.
- Целевой признак регрессии "BMI" наиболее сильно коррелирует с теми же признаками. Эти признаки обязательно следует оставить в модели регрессии.
- Большие по модулю значения коэффициентов корреляции свидетельствуют о значимой корреляции между исходными признаками и целевым признаком.  На основании корреляционной матрицы можно сделать вывод о том, что данные позволяют построить модель машинного обучения. 

## Выбор метрик для последующей оценки качества моделей. 

### В качестве метрик для решения задачи классификации  будем использовать:

Метрики, формируемые на основе матрицы ошибок:

#### Метрика precision:

Можно переводить как точность, но такой перевод совпадает с переводом метрики "accuracy".

$precision = \frac{TP}{TP+FP}$

Доля верно предсказанных классификатором положительных объектов, из всех объектов, которые классификатор верно или неверно определил как положительные.

Используется функция [precision_score.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html#sklearn.metrics.precision_score)

#### Метрика recall (полнота):

$recall = \frac{TP}{TP+FN}$

Доля верно предсказанных классификатором положительных объектов, из всех действительно положительных объектов.

Используется функция [recall_score.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html#sklearn.metrics.recall_score)

#### Метрика $F_1$-мера

Для того, чтобы объединить precision и recall в единую метрику используется $F_\beta$-мера, которая вычисляется как среднее гармоническое от precision и recall:

$F_\beta = (1+\beta^2) \cdot \frac{precision \cdot recall}{precision + recall}$

где $\beta$ определяет вес точности в метрике.

На практике чаще всего используют вариант F1-меры (которую часто называют F-мерой) при $\beta=1$:

$F_1 = 2 \cdot \frac{precision \cdot recall}{precision + recall}$

Для вычисления используется функция [f1_score.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html#sklearn.metrics.f1_score)

#### Метрика ROC AUC

Основана на вычислении следующих характеристик:

$TPR = \frac{TP}{TP+FN}$ - True Positive Rate, откладывается по оси ординат. Совпадает с recall.

$FPR = \frac{FP}{FP+TN}$ - False Positive Rate, откладывается по оси абсцисс. Показывает какую долю из объектов отрицательного класса алгоритм предсказал неверно.

Идеальная ROC-кривая проходит через точки (0,0)-(0,1)-(1,1), то есть через верхний левый угол графика.

Чем сильнее отклоняется кривая от верхнего левого угла графика, тем хуже качество классификации.

В качестве количественной метрики используется площадь под кривой - ROC AUC (Area Under the Receiver Operating Characteristic Curve). Чем ниже проходит кривая тем меньше ее площадь и тем хуже качество классификатора.

Для получения ROC AUC используется функция [roc_auc_score.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html#sklearn.metrics.roc_auc_score)

### В качестве метрик для решения задачи регрессии будем использовать:

#### [Mean absolute error](https://en.wikipedia.org/wiki/Mean_absolute_error) - средняя абсолютная ошибка

$MAE(y,\hat{y}) = \frac{1}{N} \cdot \sum\limits_{i=1}^N \lvert  y_i - \hat{y_i} \rvert $

где:
- $y$ - истинное значение целевого признака
- $\hat{y}$ - предсказанное значение целевого признака
- $N$ - размер тестовой выборки

Чем ближе значение к нулю, тем лучше качество регрессии.

Основная проблема метрики состоит в том, что она не нормирована.

Вычисляется с помощью функции [mean_absolute_error.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html#sklearn.metrics.mean_absolute_error)

#### [Mean squared error](https://en.wikipedia.org/wiki/Mean_squared_error) - средняя квадратичная ошибка

$MSE(y,\hat{y}) = \frac{1}{N} \cdot \sum\limits_{i=1}^N ( y_i - \hat{y_i} )^2 $

где:
- $y$ - истинное значение целевого признака
- $\hat{y}$ - предсказанное значение целевого признака
- $N$ - размер тестовой выборки

Вычисляется с помощью функции [mean_squared_error.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html#sklearn.metrics.mean_squared_error)

#### [Метрика $R^2$ или коэффициент детерминации](https://ru.wikipedia.org/wiki/%D0%9A%D0%BE%D1%8D%D1%84%D1%84%D0%B8%D1%86%D0%B8%D0%B5%D0%BD%D1%82_%D0%B4%D0%B5%D1%82%D0%B5%D1%80%D0%BC%D0%B8%D0%BD%D0%B0%D1%86%D0%B8%D0%B8) 

$R^2(y,\hat{y}) = 1 - \frac{\sum\limits_{i=1}^N ( y_i - \hat{y_i} )^2}{\sum\limits_{i=1}^N ( y_i - \overline{y_i} )^2} $

где:
- $y$ - истинное значение целевого признака
- $\hat{y}$ - предсказанное значение целевого признака
- $N$ - размер тестовой выборки
- $\overline{y_i} = \frac{1}{N} \cdot \sum\limits_{i=1}^N y_i $

Вычисляется с помощью функции [r2_score.](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html#sklearn.metrics.r2_score)

### Сохранение и визуализация метрик

Разработаем класс, который позволит сохранять метрики качества построенных моделей и реализует визуализацию метрик качества.

In [ ]:
class MetricLogger:
    
    def __init__(self):
        self.df = pd.DataFrame(
            {'metric': pd.Series([], dtype='str'),
            'alg': pd.Series([], dtype='str'),
            'value': pd.Series([], dtype='float')})

    def add(self, metric, alg, value):
        """
        Добавление значения
        """
        # Удаление значения если оно уже было ранее добавлено
        self.df.drop(self.df[(self.df['metric']==metric)&(self.df['alg']==alg)].index, inplace = True)
        # Добавление нового значения
        temp_df = pd.DataFrame({'metric': [metric], 'alg': [alg], 'value': [value]})
        self.df = pd.concat([self.df, temp_df], ignore_index=True)

    def get_data_for_metric(self, metric, ascending=True):
        """
        Формирование данных с фильтром по метрике
        """
        temp_data = self.df[self.df['metric']==metric]
        temp_data_2 = temp_data.sort_values(by='value', ascending=ascending)
        return temp_data_2['alg'].values, temp_data_2['value'].values
    
    def plot(self, str_header, metric, ascending=True, figsize=(5, 5)):
        """
        Вывод графика
        """
        array_labels, array_metric = self.get_data_for_metric(metric, ascending)
        fig, ax1 = plt.subplots(figsize=figsize)
        pos = np.arange(len(array_metric))
        rects = ax1.barh(pos, array_metric,
                         align='center',
                         height=0.5, 
                         tick_label=array_labels)
        ax1.set_title(str_header)
        for a,b in zip(pos, array_metric):
            plt.text(0.5, a-0.05, str(round(b,3)), color='white')
        plt.show()    

## Выбор наиболее подходящих моделей для решения задачи классификации или регрессии.

Для задачи классификации будем использовать следующие модели:
- Логистическая регрессия
- Метод ближайших соседей
- Машина опорных векторов
- Решающее дерево
- Случайный лес
- Градиентный бустинг

Для задачи регрессии будем использовать следующие модели:
- Линейная регрессия
- Метод ближайших соседей
- Машина опорных векторов
- Решающее дерево
- Случайный лес
- Градиентный бустинг

## Формирование обучающей и тестовой выборок на основе исходного набора данных.

In [ ]:
# На основе масштабированных данных выделим 
# обучающую и тестовую выборки с помощью фильтра
train_data_all = data_all[data_all['dataset']=='TRAIN']
test_data_all = data_all[data_all['dataset']=='TEST1']
train_data_all.shape, test_data_all.shape

In [ ]:
# Признаки для задачи классификации
task_clas_cols = [
    # Core features
    'Gender', 'Age', 
    'family_history_with_overweight',
    
    # Original survey items
    'FAVC', 'FCVC_scaled', 'NCP_scaled', 'CAEC_scaled', 'SMOKE', 
    'CH2O_scaled', 'SCC', 'FAF_scaled', 'TUE_scaled', 'CALC_scaled', 
    'MTRANS_scaled',
    
    # Engineered features
    'Metabolic_Risk_Score', 'Activity_Score'
]

In [ ]:
# Выборки для задачи классификации
clas_X_train = train_data_all[task_clas_cols]
clas_X_test = test_data_all[task_clas_cols]
clas_Y_train = train_data_all['Obesity']
clas_Y_test = test_data_all['Obesity']
clas_X_train.shape, clas_X_test.shape, clas_Y_train.shape, clas_Y_test.shape

In [ ]:
# Признаки для задачи регрессии
task_regr_cols = [
    # Demographic
    'Gender', 'Age', 'family_history_with_overweight',  #not height cuz otherwise the model would reverse-engineer the answer
    
    # Behavioral
    'FAVC', 'FCVC_scaled', 'NCP_scaled', 'CAEC_scaled', 'SMOKE',
    'CH2O_scaled', 'SCC', 'FAF_scaled', 'TUE_scaled', 'CALC_scaled',
    'MTRANS_scaled',
    
    # Engineered
    'Metabolic_Risk_Score', 'Activity_Score', 
    'Diet_Quality', 'Sedentary_Index', 
]

In [ ]:
# Выборки для задачи регресии
regr_X_train = train_data_all[task_regr_cols]
regr_X_test = test_data_all[task_regr_cols]
regr_Y_train = train_data_all['BMI']
regr_Y_test = test_data_all['BMI']
regr_X_train.shape, regr_X_test.shape, regr_Y_train.shape, regr_Y_test.shape

## Построение базового решения (baseline) для выбранных моделей без подбора гиперпараметров. Производится обучение моделей на основе обучающей выборки и оценка качества моделей на основе тестовой выборки.

### Решение задачи классификации

In [ ]:
# Модели
clas_models = {'LogR': LogisticRegression(max_iter=1000), 
               'KNN_5':KNeighborsClassifier(n_neighbors=5),
               'SVC':SVC(probability=True),
               'Tree':DecisionTreeClassifier(),
               'RF':RandomForestClassifier(),
               'GB':GradientBoostingClassifier()}

In [ ]:
# Сохранение метрик
clasMetricLogger = MetricLogger()

In [ ]:
# Отрисовка ROC-кривой
def draw_roc_curve(y_true, y_score, ax, pos_label=1, average='micro'):
    fpr, tpr, thresholds = roc_curve(y_true, y_score, 
                                     pos_label=pos_label)
    roc_auc_value = roc_auc_score(y_true, y_score, average=average)
    #plt.figure()
    lw = 2
    ax.plot(fpr, tpr, color='darkorange',
             lw=lw, label='ROC curve (area = %0.2f)' % roc_auc_value)
    ax.plot([0, 1], [0, 1], color='navy', lw=lw, linestyle='--')
    ax.set_xlim([0.0, 1.0])
    ax.set_xlim([0.0, 1.05])
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('Receiver operating characteristic')
    ax.legend(loc="lower right")

In [ ]:
def clas_train_model(model_name, model, clasMetricLogger):
    model.fit(clas_X_train, clas_Y_train)
    # Предсказание значений
    Y_pred = model.predict(clas_X_test)
    # Предсказание вероятности класса "1" для roc auc
    Y_pred_proba_temp = model.predict_proba(clas_X_test)
    Y_pred_proba = Y_pred_proba_temp[:,1]
    
    precision = precision_score(clas_Y_test.values, Y_pred)
    recall = recall_score(clas_Y_test.values, Y_pred)
    f1 = f1_score(clas_Y_test.values, Y_pred)
    roc_auc = roc_auc_score(clas_Y_test.values, Y_pred_proba)
    
    clasMetricLogger.add('precision', model_name, precision)
    clasMetricLogger.add('recall', model_name, recall)
    clasMetricLogger.add('f1', model_name, f1)
    clasMetricLogger.add('roc_auc', model_name, roc_auc)

    fig, ax = plt.subplots(ncols=2, figsize=(10,5))    
    draw_roc_curve(clas_Y_test.values, Y_pred_proba, ax[0])

    # Updated confusion matrix plotting
    cm = confusion_matrix(clas_Y_test.values, Y_pred, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['0','1'])
    disp.plot(cmap=plt.cm.Blues, ax=ax[1])
    
    fig.suptitle(model_name)
    plt.show()

In [ ]:
for model_name, model in clas_models.items():
    clas_train_model(model_name, model, clasMetricLogger)

### Решение задачи регрессии

In [ ]:
# Модели
regr_models = {'LR': LinearRegression(), 
               'KNN_5':KNeighborsRegressor(n_neighbors=5),
               'SVR':SVR(),
               'Tree':DecisionTreeRegressor(),
               'RF':RandomForestRegressor(),
               'GB':GradientBoostingRegressor()}

In [ ]:
# Сохранение метрик
regrMetricLogger = MetricLogger()

In [ ]:
def regr_train_model(model_name, model, regrMetricLogger):
    model.fit(regr_X_train, regr_Y_train)
    Y_pred = model.predict(regr_X_test)
    
    mae = mean_absolute_error(regr_Y_test, Y_pred)
    mse = mean_squared_error(regr_Y_test, Y_pred)
    r2 = r2_score(regr_Y_test, Y_pred)

    regrMetricLogger.add('MAE', model_name, mae)
    regrMetricLogger.add('MSE', model_name, mse)
    regrMetricLogger.add('R2', model_name, r2)    
    
    print('{} \t MAE={}, MSE={}, R2={}'.format(
        model_name, round(mae, 3), round(mse, 3), round(r2, 3)))

In [ ]:
for model_name, model in regr_models.items():
    regr_train_model(model_name, model, regrMetricLogger)

## Подбор гиперпараметров для выбранных моделей. 

In [ ]:
# Function to perform GridSearchCV and plot results
def optimize_and_plot(model, param_grid, X_train, y_train, scoring, model_name, task_type):
    gs = GridSearchCV(model, param_grid, cv=5, scoring=scoring, n_jobs=-1, verbose=1)
    gs.fit(X_train, y_train)
    
    print(f"Best parameters for {model_name}: {gs.best_params_}")
    print(f"Best {scoring} score: {gs.best_score_:.4f}")
    
    # Plot performance vs parameter values
    plt.figure(figsize=(10, 6))
    if len(param_grid) == 1:  # Single parameter case
        param_name = list(param_grid.keys())[0]
        param_values = param_grid[param_name]
        
        # Handle cases where we can't plot all parameters
        if len(str(param_values[0])) < 50:  # Avoid plotting very complex parameter values
            plt.plot(param_values, gs.cv_results_['mean_test_score'], 'o-')
            plt.xlabel(param_name)
            plt.ylabel(f'Mean {scoring} score')
            plt.title(f'{model_name} performance vs {param_name}')
        else:
            print(f"Cannot plot for parameter {param_name} - values too complex")
    else:
        print("Multi-dimensional parameter space - skipping plot")
    
    plt.show()
    return gs.best_estimator_

### Для задачи классификации

In [ ]:
clas_param_grids = {
    'LogR': {'C': np.logspace(-3, 3, 7), 'penalty': ['l2']},
    'KNN_5': {'n_neighbors': range(1, 100)},
    'SVC': [
        {'C': [0.1, 1, 10], 'kernel': ['linear']},
        {'C': [0.1, 1, 10], 'gamma': [0.1, 0.01, 0.001], 'kernel': ['rbf']}
    ],
    'Tree': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'RF': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7, None],
        'min_samples_split': [2, 5, 10]
    },
    'GB': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5],
        'max_depth': [3, 5, 7]
    }
}

In [ ]:
%%time
clas_best_models = {}
for model_name, model in clas_models.items():
    print(f"\nOptimizing {model_name}...")
    best_model = optimize_and_plot(
        model, 
        clas_param_grids[model_name], 
        clas_X_train, 
        clas_Y_train, 
        'roc_auc', 
        model_name,
        'classification'
    )
    clas_best_models[model_name + '_optimized'] = best_model

In [ ]:
# Изменение качества на тестовой выборке в зависимости от К-соседей
#plt.plot(n_range, clf_gs.cv_results_['mean_test_score'])

### Пример для задачи регрессии

In [ ]:
regr_param_grids = {
    'LR': {'fit_intercept': [True, False]},
    'KNN_5': {'n_neighbors': range(1, 31)},
    'SVR': [
        {'C': [0.1, 1, 10], 'kernel': ['linear']},
        {'C': [0.1, 1, 10], 'gamma': [0.1, 0.01, 0.001], 'kernel': ['rbf']}
    ],
    'Tree': {'max_depth': [3, 5, 7, 10, None], 'min_samples_split': [2, 5, 10]},
    'RF': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7, None],
        'min_samples_split': [2, 5, 10]
    },
    'GB': {
        'n_estimators': [50, 100, 200],
        'learning_rate': [0.01, 0.1, 0.5],
        'max_depth': [3, 5, 7]
    }
}

In [ ]:
%%time
regr_best_models = {}
for model_name, model in regr_models.items():
    print(f"\nOptimizing {model_name}...")
    best_model = optimize_and_plot(
        model, 
        regr_param_grids[model_name], 
        regr_X_train, 
        regr_Y_train, 
        'neg_mean_squared_error', 
        model_name,
        'regression'
    )
    regr_best_models[model_name + '_optimized'] = best_model

In [ ]:
# Изменение качества на тестовой выборке в зависимости от К-соседей
#plt.plot(n_range, regr_gs.cv_results_['mean_test_score'])

## Повторение пункта 8 для найденных оптимальных значений гиперпараметров. Сравнение качества полученных моделей с качеством baseline-моделей.

### Решение задачи классификации

In [ ]:
# Train and evaluate optimized models
print("\nEvaluating optimized classification models:")
for model_name, model in clas_best_models.items():
    clas_train_model(model_name, model, clasMetricLogger)

### Решение задачи регрессии

In [ ]:
print("\nEvaluating optimized regression models:")
for model_name, model in regr_best_models.items():
    regr_train_model(model_name, model, regrMetricLogger)

### Сравнение качества моделей

In [ ]:
def compare_metrics(original_models, optimized_models, metric_logger, metric_name):
    original_scores = []
    optimized_scores = []
    model_names = []
    
    # Get all metrics data for the specified metric
    metric_data = metric_logger.df[metric_logger.df['metric'] == metric_name]
    
    for model_name in original_models.keys():
        optimized_name = model_name + '_optimized'
        
        # Get original model score
        original_score = metric_data[metric_data['alg'] == model_name]['value']
        if not original_score.empty:
            original_scores.append(original_score.values[0])
        else:
            continue  # Skip if original model not found
            
        # Get optimized model score
        optimized_score = metric_data[metric_data['alg'] == optimized_name]['value']
        if not optimized_score.empty:
            optimized_scores.append(optimized_score.values[0])
            model_names.append(model_name)
    
    if not model_names:  # No data to compare
        print(f"No comparison data available for metric: {metric_name}")
        return
    
    # Create comparison plot
    x = range(len(model_names))
    plt.figure(figsize=(12, 6))
    plt.bar(x, original_scores, width=0.4, label='Original')
    plt.bar([i + 0.4 for i in x], optimized_scores, width=0.4, label='Optimized')
    plt.xticks([i + 0.2 for i in x], model_names, rotation=45)
    plt.ylabel(metric_name)
    plt.title(f'Comparison of {metric_name} between original and optimized models')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
compare_metrics(clas_models, clas_best_models, clasMetricLogger, 'roc_auc')
compare_metrics(clas_models, clas_best_models, clasMetricLogger, 'f1')

In [ ]:
compare_metrics(regr_models, regr_best_models, regrMetricLogger, 'R2')
compare_metrics(regr_models, regr_best_models, regrMetricLogger, 'MAE')

## Формирование выводов о качестве построенных моделей на основе выбранных метрик.

### Решение задачи классификации

In [ ]:
# Метрики качества модели
clas_metrics = clasMetricLogger.df['metric'].unique()
clas_metrics

In [ ]:
# Построим графики метрик качества модели
for metric in clas_metrics:
    clasMetricLogger.plot('Метрика: ' + metric, metric, figsize=(7, 6))

**Вывод: на основании трех метрик из четырех используемых,  лучшей оказались модели градиентного бустинга и случайного леса. В некоторых случаях лучшей могло оказаться дерево решений**

### Решение задачи регрессии

In [ ]:
# Метрики качества модели
regr_metrics = regrMetricLogger.df['metric'].unique()
regr_metrics

In [ ]:
regrMetricLogger.plot('Метрика: ' + 'MAE', 'MAE', ascending=False, figsize=(7, 6))

In [ ]:
regrMetricLogger.plot('Метрика: ' + 'MSE', 'MSE', ascending=False, figsize=(7, 6))

In [ ]:
regrMetricLogger.plot('Метрика: ' + 'R2', 'R2', ascending=True, figsize=(7, 6))

**Вывод: лучшими оказались модели на основе случайного леса, градиентного бустинга, решающего дерева.**

## Самые важные признаки

### Регрессия - Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor().fit(regr_X_train, regr_Y_train)
Y_pred = gb.predict(regr_X_test)

In [ ]:
# Get feature importance
feature_importance = gb.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5

# Plot
plt.figure(figsize=(10, 6))
plt.barh(pos, feature_importance[sorted_idx], align='center')
plt.yticks(pos, np.array(regr_X_train.columns)[sorted_idx])
plt.title('Gradient Boosting Feature Importance (BMI Prediction)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

# Print top 5 features
print("Top 5 Features for Gradient Boosting:")
for i in sorted_idx[-5:][::-1]:
    print(f"{regr_X_train.columns[i]}: {feature_importance[i]:.4f}")

### Классификация - Gradient Boosting

In [ ]:
gbc = GradientBoostingClassifier().fit(clas_X_train, clas_Y_train)
# Предсказание значений
Y_pred = gbc.predict(clas_X_test)

In [ ]:
# Get feature importance
feature_importance = gbc.feature_importances_
sorted_idx = np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5

# Plot
plt.figure(figsize=(10, 6))
plt.barh(pos, feature_importance[sorted_idx], align='center')
plt.yticks(pos, np.array(clas_X_train.columns)[sorted_idx])
plt.title('Gradient Boosting Feature Importance (Obesity)')
plt.xlabel('Relative Importance')
plt.tight_layout()
plt.show()

# Print top 5 features
print("Top 5 Features for Gradient Boosting:")
for i in sorted_idx[-5:][::-1]:
    print(f"{clas_X_train.columns[i]}: {feature_importance[i]:.4f}")

## **Интерпретация важности признаков в исследовании ожирения**

Полученные результаты показывают, какие факторы наиболее сильно влияют на **индекс массы тела (BMI)** и **классификацию ожирения** согласно модели градиентного бустинга. Давайте разберём их в контексте реальных медицинских и социальных тенденций.

---

### **1. Регрессия (BMI) – Что влияет на индекс массы тела?**
#### **Топ-5 признаков:**
1. **`family_history_with_overweight` (0.2212)** – **Семейная история лишнего веса**  
   - **Почему важна?** Генетическая предрасположенность играет ключевую роль в метаболизме и склонности к набору веса.  
   - **Реальная жизнь:** Дети, чьи родители страдают ожирением, имеют повышенный риск из-за наследственности и схожих пищевых привычек.  

2. **`FCVC_scaled` (0.1549)** – **Частота потребления овощей (Frequent Consumption of Vegetables)**  
   - **Почему важна?** Овощи богаты клетчаткой, снижают калорийность рациона. Низкое потребление → риск ожирения.  
   - **Реальная жизнь:** В развитых странах дешевые высококалорийные продукты вытесняют овощи, что ведёт к росту BMI.  

3. **`Age` (0.0998)** – **Возраст**  
   - **Почему важен?** С возрастом метаболизм замедляется, мышечная масса уменьшается, а жировая растёт.  
   - **Реальная жизнь:** После 30 лет люди набирают вес, если не корректируют питание и активность.  

4. **`TUE_scaled` (0.0992)** – **Время за гаджетами (Time Using Electronic Devices)**  
   - **Почему важно?** Сидячий образ жизни снижает расход калорий.  
   - **Реальная жизнь:** Офисная работа, соцсети и Netflix сокращают физическую активность.  

5. **`CAEC_scaled` (0.0992)** – **Частота перекусов (Consumption of Food Between Meals)**  
   - **Почему важно?** Частые перекусы (особенно сладким/фастфудом) ведут к избытку калорий.  
   - **Реальная жизнь:** Перекусы на работе/учебе – причина незаметного набора веса.  

#### **Вывод по BMI:**  
Главные факторы – **генетика + низкое потребление овощей + возраст + малоподвижность + перекусы**. Это соответствует современным исследованиям: ожирение связано не только с едой, но и с образом жизни и наследственностью.  

---

### **2. Классификация (Obesity) – Что определяет диагноз ожирения?**
#### **Топ-5 признаков:**
1. **`family_history_with_overweight` (0.2557)** – **Семейная история**  
   - Ещё более важна, чем для BMI, так как наследственность напрямую связана с тяжелыми формами ожирения.  

2. **`CAEC_scaled` (0.2206)** – **Перекусы**  
   - У людей с ожирением часто наблюдается **неконтролируемое потребление пищи** (в т.ч. ночные перекусы).  

3. **`Age` (0.1421)** – **Возраст**  
   - Ожирение чаще развивается у людей **30+**, когда метаболизм замедляется.  

4. **`MTRANS_scaled` (0.0701)** – **Транспорт (например, автомобиль вместо ходьбы)**  
   - Люди, которые **редко ходят пешком**, больше подвержены ожирению.  

5. **`Metabolic_Risk_Score` (0.0518)** – **Метаболический риск (например, инсулинорезистентность)**  
   - Ожирение часто сопровождается **нарушением обмена веществ**, что усугубляет проблему.  

#### **Вывод по классификации ожирения:**  
Основные факторы – **генетика + пищевое поведение (перекусы) + возраст + низкая подвижность + метаболические нарушения**. Это объясняет, почему некоторые люди, даже питаясь "как все", быстрее набирают вес – из-за наследственности и особенностей метаболизма.  

---

### **Общий вывод:**
1. **Генетика (`family_history`) – самый важный фактор.** Это объясняет, почему в некоторых семьях ожирение встречается чаще.  
2. **Пищевые привычки (`FCVC`, `CAEC`) – второй по значимости фактор.** Даже при хорошей генетике неправильное питание ведёт к ожирению.  
3. **Возраст и малоподвижность (`Age`, `TUE`, `MTRANS`) – ключевые модифицируемые факторы.** После 30 лет нужно активнее следить за питанием и спортом.  
4. **Метаболические нарушения (`Metabolic_Risk_Score`) – следствие и причина ожирения.** Порочный круг: лишний вес ухудшает метаболизм, что усугубляет ожирение.  

#### **Практические рекомендации:**
✅ **Если в семье есть случаи ожирения** – особенно важно контролировать питание и активность.  
✅ **Увеличить долю овощей в рационе** – это снижает калорийность и улучшает метаболизм.  
✅ **Бороться с перекусами** – заменить сладости на орехи/фрукты.  
✅ **Больше двигаться** – даже простая ходьба помогает.  
✅ **После 30 лет** – пересмотреть диету, добавить силовые тренировки для сохранения мышц.  

Эти данные хорошо согласуются с современной диетологией и объясняют, почему одни люди толстеют быстрее других. 